In [1]:
import os
model_dir = os.path.join('task2', 'models')

In [2]:
import torch

if torch.cuda.is_available():
    print("CUDA is available!")
    DEVICE = torch.device("cuda")
else:
    print("CUDA is not available!")
    DEVICE = torch.device("cpu")

CUDA is available!


## Import and show training dataset

In [3]:
import pickle as pkl
import numpy as np
from torch.utils.data import StackDataset, DataLoader

DATASET_PATHS = os.path.join('..','data')

TEST_DATASET_PATH = os.path.join(DATASET_PATHS,'test.pickle')
TRAIN_DATASET_PATH = os.path.join(DATASET_PATHS, 'train.pickle')

with open(TRAIN_DATASET_PATH, "rb") as f:
    train_dataset = pkl.load(f)

with open(TEST_DATASET_PATH, "rb") as f:
    test_dataset = pkl.load(f)

train_data = torch.tensor(
    np.array(list(train_dataset["sensor_data"]), dtype=np.float32),
)
train_data = train_data.moveaxis(-1,-2)

train_labels = torch.tensor(
    np.array(list(train_dataset["label"])),
)

test_data = torch.tensor(
    np.array(list(test_dataset["sensor_data"]), dtype=np.float32),
    device=DEVICE
)
test_data = test_data.moveaxis(-1,-2)

test_labels = torch.tensor(
    np.array(list(test_dataset["label"])),
    device=DEVICE
)

train_dataset = StackDataset(train_data, train_labels)
train_dataloader = DataLoader(train_dataset, batch_size=32)

test_dataset = StackDataset(test_data, test_labels)
test_dataloader = DataLoader(test_dataset, batch_size=32)

/tmp/ipykernel_449/2259471756.py:11: DeprecationWarning: numpy.core.numeric is deprecated and has been renamed to numpy._core.numeric. The numpy._core namespace contains private NumPy internals and its use is discouraged, as NumPy internals can change without warning in any release. In practice, most real-world usage of numpy.core is to access functionality in the public NumPy API. If that is the case, use the public NumPy API. If not, you are using NumPy internals. If you would still like to access an internal attribute, use numpy._core.numeric._frombuffer.
  train_dataset = pkl.load(f)
/tmp/ipykernel_449/2259471756.py:14: DeprecationWarning: numpy.core.numeric is deprecated and has been renamed to numpy._core.numeric. The numpy._core namespace contains private NumPy internals and its use is discouraged, as NumPy internals can change without warning in any release. In practice, most real-world usage of numpy.core is to access functionality in the public NumPy API. If that is the case,

## Benchmark without artificial data

In [4]:
from task1.cnn.model import CNNClassifier
import torch.nn as nn
import torch.optim as optim
from utils import benchmark

Classifier = CNNClassifier
Optimizer = optim.Adam
LossFunction = nn.CrossEntropyLoss
epochs = 7

losses, accs = benchmark(
    Classifier,
    Optimizer,
    LossFunction,
    epochs,
    train_dataloader,
    test_dataloader,
    DEVICE,
    optimizer_args={ 'lr' : 0.001 },
    log=False,
    runs=10
)

print('loss: ', np.mean(losses))
print('accs: ', np.mean(accs))

Starting run 1...


/root/.venv/lib/python3.12/site-packages/torch/nn/modules/linear.py:125: UserWarning: Attempting to use hipBLASLt on an unsupported architecture! Overriding blas backend to hipblas (Triggered internally at ../aten/src/ATen/Context.cpp:296.)
  return F.linear(input, self.weight, self.bias)


Run 1: loss 2.044306920363115 acc 0.4675324675324675
Starting run 2...
Run 2: loss 2.1353667937554084 acc 0.3986013986013986
Starting run 3...
Run 3: loss 1.974674790770143 acc 0.5364635364635365
Starting run 4...
Run 4: loss 2.54571210004233 acc 0.3906093906093906
Starting run 5...
Run 5: loss 2.0873483794552463 acc 0.4355644355644356
Starting run 6...
Run 6: loss 1.844657284634692 acc 0.5924075924075924
Starting run 7...
Run 7: loss 1.9876818914156218 acc 0.5314685314685315
Starting run 8...
Run 8: loss 2.7167975246370375 acc 0.43356643356643354
Starting run 9...
Run 9: loss 1.9309748183716309 acc 0.5634365634365635
Starting run 10...
Run 10: loss 2.15337157439995 acc 0.5924075924075924
loss:  2.142089207784518
accs:  0.4942057942057943


## Benchmark with artificial data from GAN

In [5]:
def gan_generate(model, label_dist, device):
    """
    Generates for every classification the specified number of synthetic data in `label_dist`. 
    `label_dist` is a mapping which should return an *integer* number of generated output.
    The result is a tuple of `torch.Tensor`s with the first one being the generated time series and the second one the labels for each time series.
    """
    model.eval()
    model.to(device)

    all_samples = tuple()
    all_labels = tuple()
    
    for i in range(12):
        all_samples += ( model(model.sample(batch_size=label_dist[i], device=device), labels=i*torch.ones(label_dist[i], device=device, dtype=torch.long) ), )
        all_labels += ( i*torch.ones(label_dist[i], device=device, dtype=torch.long),)
    return torch.cat(all_samples), torch.cat(all_labels)

### Import the data

In [6]:
from task2.gan.model import CNNGenerator

gan_generator = CNNGenerator()
gan_generator.to(DEVICE)
gan_generator.load_state_dict(torch.load(os.path.join(model_dir,'generator-gan.pt'), weights_only=True))

<All keys matched successfully>

### Choose a distribution

Just for testing I choose a constant number of samples per label. Here we can adjust the distribution if we want to fill up with artificial data and so on.

In [7]:

# synth_distribution = np.max(distribution) - distribution # fill any inbalance with synthetic data 
# _, synth_distribution = np.unique(label_data, return_counts=True) 
# synth_distribution = synth_distribution // 2 # maybe reduce this by 1/2
synth_distribution = 100*np.ones(12, dtype=int) # just take 100 from everything
print(synth_distribution)

[100 100 100 100 100 100 100 100 100 100 100 100]


### Generate the data

In [8]:
from torch.utils.data import StackDataset

# generate it
gan_synthetic_data, gan_synthetic_labels = gan_generate(gan_generator, synth_distribution, DEVICE)
# move back to cpu
gan_synthetic_data = gan_synthetic_data.cpu().detach()
gan_synthetic_labels = gan_synthetic_labels.cpu().detach()

gan_synthetic_dataset = StackDataset(gan_synthetic_data.moveaxis(-1,-2), gan_synthetic_labels)

# how much generated data do we actually have?
num_synthetic_data = gan_synthetic_data.shape[0]

### Show the data

this may not work in VSCode

In [9]:
# show the generated data
from ipywidgets import interact, IntSlider
from utils import show_dataset

FAILURE_CLASSES = [
    ("0-0-0-0", "All brakes intact (no irregularities)."),
    ("0-0-1-0", "Rear Left Brake with irregularity."),
    ("0-0-0-1", "Rear Right Brake with irregularity."),
    ("0-0-1-1", "Both Rear Brakes with irregularities."),
    ("1-0-0-0", "Front Left Brake with irregularity."),
    ("0-1-0-0", "Front Right Brake with irregularity."),
    ("1-1-0-0", "Both Front Brakes with irregularities."),
    ("1-0-1-0", "Left Brakes (Front + Rear Left) with irregularities."),
    ("0-1-0-1", "Right Brakes (Front + Rear Right) with irregularities."),
    ("1-0-0-1", "Front Left and Rear Right Brakes with irregularities."),
    ("0-1-1-0", "Front Right and Rear Left Brakes with irregularities."),
    ("1-1-1-1", "All Brakes with irregularities.")
]

@interact(index=IntSlider(min=0,max=num_synthetic_data-1,step=1,value=0))
def show(index):
    fig = show_dataset(gan_synthetic_data[index].cpu().detach().numpy())
    fig.suptitle('Synthetic data ({})'.format(FAILURE_CLASSES[int(gan_synthetic_labels[index])][0]) )

interactive(children=(IntSlider(value=0, description='index', max=1199), Output()), _dom_classes=('widget-inte…

### Run the benchmark!

In [10]:
from torch.utils.data import ConcatDataset
from utils import benchmark

gan_train_dataset = ConcatDataset((train_dataset, gan_synthetic_dataset))

losses, accs = benchmark(
    Classifier,
    Optimizer,
    LossFunction,
    epochs,
    DataLoader(gan_train_dataset, batch_size=32),
    test_dataloader,
    DEVICE,
    optimizer_args={ 'lr' : 0.001 },
    log=False,
    runs=5
)

print('loss: ', np.mean(losses))
print('accs: ', np.mean(accs))

Starting run 1...
Run 1: loss 3.998360110805942 acc 0.2997002997002997
Starting run 2...
Run 2: loss 2.179415815002792 acc 0.43156843156843155
Starting run 3...
Run 3: loss 3.2927856614420583 acc 0.06593406593406594
Starting run 4...
Run 4: loss 2.1522121371089162 acc 0.34465534465534464
Starting run 5...
Run 5: loss 3.2401418941957014 acc 0.08591408591408592
loss:  2.972583123711082
accs:  0.24555444555444553


## Benchmark with artificial data from RNN

In [11]:
DATASET_PATHS = os.path.join('..','data')

TEST_DATASET_PATH = os.path.join(DATASET_PATHS,'test.pickle')
TRAIN_DATASET_PATH = os.path.join(DATASET_PATHS, 'train.pickle')

with open(TRAIN_DATASET_PATH, "rb") as f:
    train_dataset = pkl.load(f)

with open(TEST_DATASET_PATH, "rb") as f:
    test_dataset = pkl.load(f)

train_data = torch.tensor(
    np.array(list(train_dataset["sensor_data"]), dtype=np.float32),
)
train_data = train_data.moveaxis(-1,-2)

train_labels = torch.tensor(
    np.array(list(train_dataset["label"])),
)

test_data = torch.tensor(
    np.array(list(test_dataset["sensor_data"]), dtype=np.float32),
    device=DEVICE
)
test_data = test_data.moveaxis(-1,-2)

test_labels = torch.tensor(
    np.array(list(test_dataset["label"])),
    device=DEVICE
)

train_dataset = StackDataset(train_data, train_labels)
train_dataloader = DataLoader(train_dataset, batch_size=32)

test_dataset = StackDataset(test_data, test_labels)
test_dataloader = DataLoader(test_dataset, batch_size=32)

/tmp/ipykernel_449/1161956199.py:7: DeprecationWarning: numpy.core.numeric is deprecated and has been renamed to numpy._core.numeric. The numpy._core namespace contains private NumPy internals and its use is discouraged, as NumPy internals can change without warning in any release. In practice, most real-world usage of numpy.core is to access functionality in the public NumPy API. If that is the case, use the public NumPy API. If not, you are using NumPy internals. If you would still like to access an internal attribute, use numpy._core.numeric._frombuffer.
  train_dataset = pkl.load(f)
/tmp/ipykernel_449/1161956199.py:10: DeprecationWarning: numpy.core.numeric is deprecated and has been renamed to numpy._core.numeric. The numpy._core namespace contains private NumPy internals and its use is discouraged, as NumPy internals can change without warning in any release. In practice, most real-world usage of numpy.core is to access functionality in the public NumPy API. If that is the case, 

### Define the sequence generation functions

These will do similar stuff as `gan_generate`. We initialize with the mean starting point of the training data perturbed by some gaussian noise. Note here that this takes way longer than generation using a GAN.

In [12]:
# SEQUENCE GENERATION FUNCTION
def generate_sequence(generator, seed, label, device, seq_len=128):
    generator.eval()

    label = torch.tensor([label], dtype=torch.long).to(device)
    seed = torch.tensor(seed, dtype=torch.float32).unsqueeze(0).unsqueeze(1).to(device)  # (1, 1, sensor_dim)
    generated = [seed.squeeze(0)]

    hidden = None
    for _ in range(seq_len - 1):
        output, hidden = generator(seed, label, hidden)
        seed = output
        generated.append(output.squeeze(0))

    return torch.cat(generated, dim=0).cpu().detach()

def rnn_generate(generator, mean_start, std_start, label_counts, device, seq_len=128):
    sensor_dim = generator.fc.out_features

    all_sequences = []
    all_labels = []

    for label in range(len(label_counts)):
        n_samples = label_counts[label]
        for _ in range(n_samples):
            # Generate a seed around the mean start with some noise
            seed = mean_start + np.random.normal(0, std_start * 0.5, size=mean_start.shape)
            synthetic_seq = generate_sequence(generator, seed, label, device, seq_len=seq_len)
            synthetic_seq = synthetic_seq.numpy()
            all_sequences.append(synthetic_seq)
            all_labels.append(label)

    return np.array(all_sequences), np.array(all_labels)

In [13]:
from task2.rnn.model import SequenceGeneratorEmbedded
import pickle as pkl

rnn_generator = SequenceGeneratorEmbedded()

with open(os.path.join(DATASET_PATHS, 'train.pickle'), "rb") as f:
    ld = pkl.load(f)
    start_values = np.stack(ld['sensor_data'].apply(lambda x: x[0]))  # shape: (num_samples, sensor_dim)
    
mean_start = start_values.mean(axis=0)
std_start = start_values.std(axis=0)

rnn_generator.to(DEVICE)
rnn_generator.load_state_dict(torch.load(os.path.join(model_dir,'generator-rnn.pt'), weights_only=True))

rnn_synthetic_data, rnn_synthetic_labels = rnn_generate(rnn_generator, mean_start, std_start, synth_distribution, DEVICE)
rnn_synthetic_data = torch.tensor(rnn_synthetic_data)
rnn_synthetic_labels = torch.tensor(rnn_synthetic_labels)

rnn_synthetic_dataset = StackDataset( rnn_synthetic_data.moveaxis(-1,-2) ,rnn_synthetic_labels )

/tmp/ipykernel_449/2291007317.py:7: DeprecationWarning: numpy.core.numeric is deprecated and has been renamed to numpy._core.numeric. The numpy._core namespace contains private NumPy internals and its use is discouraged, as NumPy internals can change without warning in any release. In practice, most real-world usage of numpy.core is to access functionality in the public NumPy API. If that is the case, use the public NumPy API. If not, you are using NumPy internals. If you would still like to access an internal attribute, use numpy._core.numeric._frombuffer.
  ld = pkl.load(f)


### Show the computed data

In [14]:
@interact(index=IntSlider(min=0,max=num_synthetic_data-1,step=1,value=0))
def show(index):
    fig = show_dataset(rnn_synthetic_data[index].cpu().detach().numpy())
    fig.suptitle('Synthetic data ({})'.format(FAILURE_CLASSES[int(rnn_synthetic_labels[index])][0]) )

interactive(children=(IntSlider(value=0, description='index', max=1199), Output()), _dom_classes=('widget-inte…

### Run the benchmark!

In [15]:
rnn_train_dataset = ConcatDataset((train_dataset, rnn_synthetic_dataset))

losses, accs = benchmark(
    Classifier,
    Optimizer,
    LossFunction,
    epochs,
    DataLoader(rnn_train_dataset, batch_size=32),
    test_dataloader,
    DEVICE,
    optimizer_args={ 'lr' : 0.001 },
    log=False,
    runs=5
)

print('loss: ', np.mean(losses))
print('accs: ', np.mean(accs))

Starting run 1...
Run 1: loss 1.7427211783149026 acc 0.5924075924075924
Starting run 2...
Run 2: loss 3.488940687208147 acc 0.4595404595404595
Starting run 3...
Run 3: loss 3.266135004016903 acc 0.12987012987012986
Starting run 4...
Run 4: loss 3.102014280103899 acc 0.3306693306693307
Starting run 5...
Run 5: loss 2.2144948080941274 acc 0.43656343656343655
loss:  2.7628611915475956
accs:  0.38981018981018983
